# 用 Bird et al. (2016) Fig. 2 检验 HMF 与两体捕获率

这个 Notebook 不是把 Bird 论文和当前 PBH 论文当成同一个模型，而是进行交叉检验：

1. 检查 `hmf` 的 `Mmin`、`Mmax` 和质量单位；
2. 调用主体脚本 `twobodycapture.capture_rate_per_halo_a14_per_year` 计算 $z=0$ 单晕率；
3. 用 Tinker08、Press-Schechter 和 Jenkins 晕质量函数构造每个 $d\ln M$ 区间的贡献；
4. 与《Did LIGO detect dark matter?》Fig. 2 的量级和曲线形状比较。

Bird Eq. (10) 为

$$
\mathcal V=\int R_{\rm halo}(M)\frac{dn}{dM}dM
=\int R_{\rm halo}(M)\frac{dn}{d\ln M}d\ln M.
$$

因此 Fig. 2 纵轴对应 $R_{\rm halo}(M)\,dn/d\ln M$，而不是已经对所有质量积分后的单个数。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from astropy.cosmology import Planck18
from hmf import MassFunction

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "twobodycapture.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pbhmassfunction as pmf
import twobodycapture as capture

PROJECT_ROOT

## 1. 直接检查 `hmf` 的质量边界

`hmf.MassFunction` 要求

$$
M_{\min}^{\rm input}=\log_{10}\left(M_{\min}^{\rm physical}h\right),
$$

因为包内质量数组的数值单位是 $M_\odot/h$。`np.arange` 生成的上界不包含在内，所以主体函数给 `Mmax` 增加半个步长，再筛选回用户要求的物理区间。

In [ ]:
physical_mass_grid, ps_dndlnm = (
    capture.press_schechter_halo_mass_function_hmfcalc(
        z=0.0,
        minimum_halo_mass_msun=1.0e3,
        maximum_halo_mass_msun=1.0e15,
        dlog10m=0.1,
    )
)

print(f"网格点数：{physical_mass_grid.size}")
print(f"第一个物理质量：{physical_mass_grid[0]:.16e} M_sun")
print(f"最后一个物理质量：{physical_mass_grid[-1]:.16e} M_sun")
print(f"相邻点 dex：{np.log10(physical_mass_grid[1]/physical_mass_grid[0]):.6f}")

## 2. 建立 Bird Fig. 2 的质量网格和三种 HMF

Bird Fig. 2 横轴明确写成 $M_{\rm vir}(M_\odot/h)$，所以这里直接令 `MassFunction` 的对数边界为 3 到 15。返回的 `mass_hinv_msun` 用于横轴；传给主体单晕率函数前再除以 $h$，得到物理 $M_\odot$。

当前 `hmf` 配置仍使用项目已经明确记录的 Planck18、CAMB、$\sigma_8=0.8159$、$n_s=0.9667$。Bird 原始数值配置未全部公开，因此这是交叉检验，不是逐像素复原。

In [ ]:
HUBBLE_H = float(Planck18.h)
HMF_COMMON_ARGUMENTS = dict(
    Mmin=3.0,
    Mmax=15.05,
    dlog10m=0.1,
    z=0.0,
    delta_c=1.686,
    cosmo_model=Planck18,
    sigma_8=capture.HMF_SIGMA_8,
    n=capture.HMF_SPECTRAL_INDEX,
    transfer_model="CAMB",
    transfer_params={"extrapolate_with_eh": True},
)

hmf_tinker = MassFunction(hmf_model="Tinker08", **HMF_COMMON_ARGUMENTS)
hmf_press_schechter = MassFunction(hmf_model="PS", **HMF_COMMON_ARGUMENTS)
hmf_jenkins = MassFunction(hmf_model="Jenkins", **HMF_COMMON_ARGUMENTS)

mass_hinv_msun = np.asarray(hmf_press_schechter.m)
mass_physical_msun = mass_hinv_msun / HUBBLE_H

def physical_dndlnm(hmf_result):
    return np.asarray(hmf_result.dndlnm) * HUBBLE_H**3

dndlnm_tinker_mpc3 = physical_dndlnm(hmf_tinker)
dndlnm_ps_mpc3 = physical_dndlnm(hmf_press_schechter)
dndlnm_jenkins_mpc3 = physical_dndlnm(hmf_jenkins)

print(f"Bird 横轴端点：{mass_hinv_msun[0]:.3e} 到 {mass_hinv_msun[-1]:.3e} M_sun/h")
print(f"传给主体单晕率的物理端点：{mass_physical_msun[0]:.3e} 到 {mass_physical_msun[-1]:.3e} M_sun")

## 3. 调用主体脚本计算两种浓度模型的每晕率

Bird Fig. 2 使用 $30M_\odot$ 单色 PBH。这里不另写捕获公式，而是直接调用当前主体函数，从而检验现有单晕率和 HMF 组合后的结果。

In [ ]:
mono_nodes, mono_weights = pmf.monochromatic_mass_distribution(30.0)

rate_per_halo_ludlow = np.asarray([
    capture.capture_rate_per_halo_a14_per_year(
        halo_mass_msun=halo_mass_msun,
        z=0.0,
        mass_nodes_msun=mono_nodes,
        mass_probability_weights=mono_weights,
        concentration_model="ludlow16",
        f_pbh=1.0,
    )
    for halo_mass_msun in mass_physical_msun
])

rate_per_halo_prada = np.asarray([
    capture.capture_rate_per_halo_a14_per_year(
        halo_mass_msun=halo_mass_msun,
        z=0.0,
        mass_nodes_msun=mono_nodes,
        mass_probability_weights=mono_weights,
        concentration_model="prada12",
        f_pbh=1.0,
    )
    for halo_mass_msun in mass_physical_msun
])

## 4. 构造 Fig. 2 的四条曲线

按照原图及正文，蓝、绿两条曲线改变浓度关系但使用基准 Tinker HMF；红、青两条曲线固定 Ludlow 浓度并改用 Press-Schechter 和 Jenkins HMF。乘以 $10^9$ 是把每 $\mathrm{Mpc}^3$ 换成每 $\mathrm{Gpc}^3$。

In [ ]:
MPC3_PER_GPC3 = 1.0e9

curve_ludlow_tinker = (
    rate_per_halo_ludlow * dndlnm_tinker_mpc3 * MPC3_PER_GPC3
)
curve_prada_tinker = (
    rate_per_halo_prada * dndlnm_tinker_mpc3 * MPC3_PER_GPC3
)
curve_ludlow_ps = (
    rate_per_halo_ludlow * dndlnm_ps_mpc3 * MPC3_PER_GPC3
)
curve_ludlow_jenkins = (
    rate_per_halo_ludlow * dndlnm_jenkins_mpc3 * MPC3_PER_GPC3
)

for label, curve in [
    ("Ludlow + Tinker08", curve_ludlow_tinker),
    ("Prada + Tinker08", curve_prada_tinker),
    ("Ludlow + Press-Schechter", curve_ludlow_ps),
    ("Ludlow + Jenkins", curve_ludlow_jenkins),
]:
    print(
        f"{label:28s}: "
        f"dV/dlnM(1e3)={curve[0]:.4e}, "
        f"axis integral={np.trapezoid(curve, x=np.log(mass_hinv_msun)):.4e}"
    )

## 5. 绘制 Bird Fig. 2 交叉检验图

横纵轴范围、颜色和线型按原图设置。因为当前浓度关系、宇宙学和密度定义并不完全等同于 Bird 2016，验收重点是数量级、低质量主导趋势以及 Jenkins 截断，而不是逐像素重合。

In [ ]:
figure, axis = plt.subplots(figsize=(7.0, 5.2))
axis.plot(mass_hinv_msun, curve_ludlow_tinker, color="blue", linewidth=2.0, label="Ludlow concentration")
axis.plot(mass_hinv_msun, curve_prada_tinker, color="green", linestyle="--", linewidth=2.0, label="Prada concentration")
axis.plot(mass_hinv_msun, curve_ludlow_ps, color="red", linestyle=":", linewidth=2.0, label="Press-Schechter m.f.")
axis.plot(mass_hinv_msun, curve_ludlow_jenkins, color="c", linestyle="-.", linewidth=2.0, label="Jenkins m.f.")

axis.set_xscale("log")
axis.set_yscale("log")
axis.set_xlim(1.0e3, 1.0e15)
axis.set_ylim(1.0e-8, 1.0)
axis.set_xlabel(r"$M_{\rm vir}\;(M_\odot/h)$")
axis.set_ylabel(r"$d\mathcal{V}/d\ln M\;(\mathrm{yr}^{-1}\,\mathrm{Gpc}^{-3})$")
axis.legend(loc="lower left", fontsize=8.5)
axis.tick_params(which="both", direction="in", top=True, right=True)

figure.tight_layout()
figure_path = PROJECT_ROOT / "did_ligo_fig2_reproduction.png"
figure.savefig(figure_path, dpi=240, bbox_inches="tight")
print(f"图像已保存：{figure_path}")
plt.show()

In [ ]:
output_table = np.column_stack([
    mass_hinv_msun,
    mass_physical_msun,
    curve_ludlow_tinker,
    curve_prada_tinker,
    curve_ludlow_ps,
    curve_ludlow_jenkins,
])
csv_path = PROJECT_ROOT / "did_ligo_fig2_reproduction.csv"
np.savetxt(
    csv_path,
    output_table,
    delimiter=",",
    header=(
        "mass_msun_per_h,mass_physical_msun,"
        "ludlow_tinker_gpc-3_yr-1,prada_tinker_gpc-3_yr-1,"
        "ludlow_press_schechter_gpc-3_yr-1,"
        "ludlow_jenkins_gpc-3_yr-1"
    ),
    comments="",
    fmt="%.10e",
)
print(f"数据已保存：{csv_path}")

## 6. 检查积分测度与质量下限敏感性

第一项比较 $d\ln M$ 和 $d\log_{10}M$ 两种完全等价的积分写法，用来排除遗漏 $\ln 10$。第二项改变 Fig. 8 的物理质量下限，判断 $10^3M_\odot$ 是否会单独造成近一个数量级的缺口。

In [ ]:
ps_identity_error = np.max(
    np.abs(hmf_press_schechter.dndlnm - mass_hinv_msun * hmf_press_schechter.dndm)
    / np.maximum(np.abs(hmf_press_schechter.dndlnm), 1.0e-300)
)
ps_curve_per_dlog10m = (
    rate_per_halo_ludlow
    * np.asarray(hmf_press_schechter.dndlog10m)
    * HUBBLE_H**3
    * MPC3_PER_GPC3
)
integral_dlnm = np.trapezoid(curve_ludlow_ps, x=np.log(mass_hinv_msun))
integral_dlog10m = np.trapezoid(
    ps_curve_per_dlog10m, x=np.log10(mass_hinv_msun)
)

print(f"dndlnm=m*dndm 最大相对误差：{ps_identity_error:.3e}")
print(f"用 dlnM 积分：{integral_dlnm:.12e}")
print(f"用 dlog10M 积分：{integral_dlog10m:.12e}")

for redshift in (0.0, 12.0):
    print(f"\nz={redshift:.0f} 的质量下限敏感性：")
    for minimum_mass in (100.0, 400.0, 1.0e3, 1.0e4):
        total_rate = capture.comoving_capture_rate_eq18_gpc3_per_year(
            z=redshift,
            mass_nodes_msun=mono_nodes,
            mass_probability_weights=mono_weights,
            concentration_model="ludlow16",
            minimum_halo_mass_msun=minimum_mass,
            maximum_halo_mass_msun=1.0e15,
            dlog10m=0.1,
        )
        print(f"Mmin={minimum_mass:7.0f} M_sun -> R={total_rate:.6f} Gpc^-3 yr^-1")

## 7. 诊断结论

实际结果重现了 Bird Fig. 2 的主要数量级、随晕质量增加而下降的趋势，以及 Jenkins 在低质量端约四个数量级的抑制。当前曲线并非逐像素一致，尤其 Prada 小质量端偏高；这是因为本项目使用 Prada12/Ludlow16、Planck18 和临界密度定义，而 Bird 2016 的浓度版本、宇宙学及 $200\rho_{\rm mean}$ 口径并不完全相同。尽管如此，以下单位和积分链条在 $z=0$ 已得到独立支持：

$$
M_\odot/h\longrightarrow M_\odot\longrightarrow R_{\rm halo}(M,0)\longrightarrow\frac{dn}{d\ln M}\longrightarrow\frac{d\mathcal V}{d\ln M}.
$$

两种积分测度给出完全相同的数值，说明没有漏掉 $\ln10$ 或质量 Jacobian。把 Fig. 8 的下限从 $10^3M_\odot$ 降至 $400M_\odot$，在 $z=12$ 只把结果从约 $10.38$ 提高到 $17.54\,\mathrm{Gpc}^{-3}\,\mathrm{yr}^{-1}$，仍远低于论文约 $150$。因此质量边界不是主要原因；问题范围应缩小到高红移 HMF 配置、浓度演化，或论文生成 Fig. 8 时对晕质量历史的具体处理。